# SI10-2026 | Ponderada | Análise de Sensibilidade em Métricas de Interface Digital

Nesta atividade, você vai analisar quais variáveis de uma interface digital têm maior impacto sobre a taxa de conversão.

A entrega deve ser feita neste notebook, com código, tabelas, gráficos e respostas curtas.

## Contexto

Uma equipe de produto quer decidir qual métrica de interface deve receber prioridade no próximo ciclo de melhoria.

Os dados representam observações diárias de um aplicativo de compras.

A métrica alvo é a taxa de conversão.

As variáveis de entrada são taxa de abandono do carrinho, profundidade média de scroll e tempo até o primeiro clique em produto.

## Preparação

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.precision", 3)


## Dados

Execute a célula abaixo para criar a base da atividade.

In [2]:
rng = np.random.default_rng(42)
n_dias = 180

taxa_abandono = rng.normal(48, 8, n_dias).clip(25, 75)
profundidade_scroll = rng.normal(62, 12, n_dias).clip(25, 95)
tempo_primeiro_clique = rng.normal(7, 2.2, n_dias).clip(2, 15)

ruido = rng.normal(0, 0.35, n_dias)
taxa_conversao = (
    7.5
    - 0.055 * taxa_abandono
    + 0.026 * profundidade_scroll
    - 0.085 * tempo_primeiro_clique
    + ruido
).clip(0.5, 9.0)

df = pd.DataFrame({
    "data": pd.date_range("2026-01-01", periods=n_dias, freq="D"),
    "taxa_abandono_carrinho_pct": taxa_abandono,
    "profundidade_scroll_pct": profundidade_scroll,
    "tempo_primeiro_clique_s": tempo_primeiro_clique,
    "taxa_conversao_pct": taxa_conversao,
})

df.head()

,data,taxa_abandono_carrinho_pct,profundidade_scroll_pct,tempo_primeiro_clique_s,taxa_conversao_pct
0,2026-01-01,50.438,77.672,6.664,5.589
1,2026-01-02,39.680,64.633,7.843,6.042
2,2026-01-03,54.004,57.069,9.200,5.318
3,2026-01-04,55.525,75.275,4.671,5.944
4,2026-01-05,32.392,67.145,6.725,6.804


## Parte 1: Exploração

Crie ao menos um gráfico ou tabela para investigar a relação entre as variáveis de entrada e a taxa de conversão.

In [3]:
# Análise exploratória com tabela-resumo e correlação com a taxa de conversão.
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

colunas_numericas = features + [target]

resumo_variaveis = (
    df[colunas_numericas]
    .describe()
    .T[["mean", "std", "min", "25%", "50%", "75%", "max"]]
    .rename(columns={
        "mean": "média",
        "std": "desvio_padrão",
        "min": "mínimo",
        "25%": "p25",
        "50%": "mediana",
        "75%": "p75",
        "max": "máximo",
    })
)

correlacoes_target = (
    df[colunas_numericas]
    .corr()[target]
    .drop(target)
    .sort_values(key=lambda s: s.abs(), ascending=False)
    .reset_index()
    .rename(columns={"index": "variável", target: "correlação_com_conversão"})
)

display(resumo_variaveis)
display(correlacoes_target)


,média,desvio_padrão,mínimo,p25,mediana,p75,máximo
taxa_abandono_carrinho_pct,47.546,6.939,30.944,42.697,47.466,51.916,71.311
profundidade_scroll_pct,62.449,12.166,31.200,54.099,62.511,69.722,95.000
tempo_primeiro_clique_s,6.971,2.244,2.000,5.477,7.072,8.466,12.715
taxa_conversao_pct,5.869,0.645,4.352,5.434,5.916,6.307,7.753


,variável,correlação_com_conversão
0,taxa_abandono_carrinho_pct,-0.643
1,profundidade_scroll_pct,0.485
2,tempo_primeiro_clique_s,-0.229


In [4]:
# Visualização das relações entre cada variável de entrada e a taxa de conversão.
for variavel_x in features:
    fig = px.scatter(
        df,
        x=variavel_x,
        y=target,
        title=f"Relação entre {variavel_x} e taxa de conversão",
        opacity=0.75,
    )
    fig.show()

# Leitura por quartis: mostra como a conversão média muda nos níveis baixos/altos de cada variável.
analise_quartis = []
for variavel in features:
    grupos = pd.qcut(df[variavel], q=4, labels=["Q1 baixo", "Q2", "Q3", "Q4 alto"])
    tabela = (
        df.assign(quartil=grupos)
        .groupby("quartil", observed=True)
        .agg(
            valor_médio_variável=(variavel, "mean"),
            conversão_média_pct=(target, "mean"),
            observações=(target, "size"),
        )
        .reset_index()
    )
    tabela.insert(0, "variável", variavel)
    analise_quartis.append(tabela)

analise_quartis = pd.concat(analise_quartis, ignore_index=True)
analise_quartis


,variável,quartil,valor_médio_variável,conversão_média_pct,observações
0,taxa_abandono_carrinho_pct,Q1 baixo,38.847,6.410,45
1,taxa_abandono_carrinho_pct,Q2,45.099,5.964,45
2,taxa_abandono_carrinho_pct,Q3,49.914,5.722,45
3,taxa_abandono_carrinho_pct,Q4 alto,56.323,5.380,45
4,profundidade_scroll_pct,Q1 baixo,46.974,5.481,45
5,profundidade_scroll_pct,Q2,58.672,5.641,45
6,profundidade_scroll_pct,Q3,65.807,6.063,45
7,profundidade_scroll_pct,Q4 alto,78.344,6.291,45
8,tempo_primeiro_clique_s,Q1 baixo,4.052,6.040,45
9,tempo_primeiro_clique_s,Q2,6.297,5.912,45


Escreva quais duas variáveis você escolheu para a análise de sensibilidade e justifique com evidências da exploração.

**Resposta:**

Escolhi **taxa_abandono_carrinho_pct** e **profundidade_scroll_pct** para a análise de sensibilidade. A escolha foi baseada principalmente na correlação com a taxa de conversão: a taxa de abandono apresentou a relação mais forte e negativa com a conversão, enquanto a profundidade de scroll apresentou uma relação positiva relevante. Isso significa que, quando o abandono aumenta, a conversão tende a cair; quando o scroll médio aumenta, a conversão tende a subir.

A leitura por quartis reforça essa decisão, porque permite comparar dias com níveis baixos e altos de cada variável. Nos dias com maior abandono, a conversão média tende a ser menor. Já nos dias com maior profundidade de scroll, a conversão média tende a ser maior. Assim, as duas variáveis parecem mais úteis para orientar uma decisão de produto do que analisar apenas o tempo até o primeiro clique, que também importa, mas aparece com menor força relativa na exploração inicial.


In [5]:
# Gráfico extra para deixar clara a força e a direção das relações exploratórias.
fig = px.bar(
    correlacoes_target,
    x="variável",
    y="correlação_com_conversão",
    color="correlação_com_conversão",
    title="Correlação das variáveis de interface com a taxa de conversão",
    text="correlação_com_conversão",
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.show()


## Parte 2: Modelo

Ajuste o modelo abaixo para estimar a taxa de conversão a partir das variáveis de entrada.

In [6]:
features = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
    "tempo_primeiro_clique_s",
]
target = "taxa_conversao_pct"

X = df[features].to_numpy()
y = df[target].to_numpy()

X_design = np.column_stack([np.ones(len(X)), X])

coeficientes, *_ = np.linalg.lstsq(X_design, y, rcond=None)

pred = X_design @ coeficientes
erro = y - pred

mae = np.mean(np.abs(erro))
rmse = np.sqrt(np.mean(erro ** 2))
r2 = 1 - (np.sum(erro ** 2) / np.sum((y - y.mean()) ** 2))
mape = np.mean(np.abs(erro / y)) * 100

tabela_coeficientes = pd.DataFrame({
    "termo": ["intercepto"] + features,
    "coeficiente": coeficientes,
})

tabela_metricas = pd.DataFrame({
    "métrica": ["MAE", "RMSE", "R²", "MAPE"],
    "valor": [mae, rmse, r2, mape],
    "interpretação_curta": [
        "erro médio absoluto em pontos percentuais de conversão",
        "erro quadrático médio, mais sensível a erros maiores",
        "proporção da variação da conversão explicada pelo modelo",
        "erro percentual médio em relação à conversão observada",
    ],
})

display(tabela_coeficientes)
display(tabela_metricas)


,termo,coeficiente
0,intercepto,7.883
1,taxa_abandono_carrinho_pct,-0.060
2,profundidade_scroll_pct,0.024
3,tempo_primeiro_clique_s,-0.094


,métrica,valor,interpretação_curta
0,MAE,0.276,erro médio absoluto em pontos percentuais de c...
1,RMSE,0.344,"erro quadrático médio, mais sensível a erros m..."
2,R²,0.714,proporção da variação da conversão explicada p...
3,MAPE,4.767,erro percentual médio em relação à conversão o...


Interprete o erro do modelo em relação à taxa de conversão.

**Resposta:**

O modelo teve erro baixo em relação à escala da taxa de conversão. O **MAE ficou em torno de 0,28 ponto percentual**, o que significa que, em média, a previsão do modelo erra aproximadamente 0,28 p.p. de conversão por dia. O **RMSE ficou em torno de 0,34 p.p.**, um pouco maior porque penaliza mais os erros grandes. Como a conversão média está perto de **5,87%**, esse erro é relativamente pequeno para uma análise exploratória de decisão.

Além disso, o **R² ficou perto de 0,71**, indicando que o modelo explica uma parte relevante da variação observada na taxa de conversão. Ele não deve ser tratado como prova causal, mas é bom o suficiente para apoiar uma análise de sensibilidade, porque os sinais dos coeficientes fazem sentido para o contexto: abandono reduz conversão, scroll aumenta conversão e maior tempo até o primeiro clique reduz conversão.


In [7]:
# Validação extra: separação temporal entre treino e teste.
# Como os dados representam dias, o teste usa os últimos 20% dos dias para simular previsão futura.
ponto_corte = int(len(df) * 0.8)

treino = df.iloc[:ponto_corte].copy()
teste = df.iloc[ponto_corte:].copy()

X_treino = np.column_stack([np.ones(len(treino)), treino[features].to_numpy()])
y_treino = treino[target].to_numpy()

X_teste = np.column_stack([np.ones(len(teste)), teste[features].to_numpy()])
y_teste = teste[target].to_numpy()

coef_treino, *_ = np.linalg.lstsq(X_treino, y_treino, rcond=None)
pred_teste = X_teste @ coef_treino
erro_teste = y_teste - pred_teste

metricas_validacao = pd.DataFrame({
    "amostra": ["base completa", "teste temporal últimos 20%"],
    "MAE": [mae, np.mean(np.abs(erro_teste))],
    "RMSE": [rmse, np.sqrt(np.mean(erro_teste ** 2))],
})

df_modelo = df.assign(previsão_pct=pred, erro_pct=erro)

display(metricas_validacao)

fig = px.scatter(
    df_modelo,
    x="previsão_pct",
    y="erro_pct",
    title="Análise de resíduos: erro do modelo por valor previsto",
    labels={"previsão_pct": "taxa de conversão prevista (%)", "erro_pct": "erro observado - previsto (p.p.)"},
)
fig.add_hline(y=0)
fig.show()


,amostra,MAE,RMSE
0,base completa,0.276,0.344
1,teste temporal últimos 20%,0.264,0.325


## Parte 3: Análise de Sensibilidade

Calcule a sensibilidade para duas variáveis de entrada usando uma variação de 10%.

Use a fórmula: sensibilidade igual à variação percentual da saída dividida pela variação percentual da entrada.

In [8]:
def prever_linha(linha):
    entrada = np.array([1] + [linha[feature] for feature in features])
    return float(entrada @ coeficientes)


linha_base = df[features].mean().to_dict()
saida_base = prever_linha(linha_base)

linha_base, saida_base

({'taxa_abandono_carrinho_pct': 47.54587889307049,
  'profundidade_scroll_pct': 62.44918010647032,
  'tempo_primeiro_clique_s': 6.9708957453209095},
 5.868747841831934)

In [9]:
# Variáveis escolhidas na Parte 1.
variaveis_escolhidas = [
    "taxa_abandono_carrinho_pct",
    "profundidade_scroll_pct",
]

if len(variaveis_escolhidas) != 2:
    raise ValueError("Preencha variaveis_escolhidas com duas variáveis da lista features.")

variaveis_invalidas = [v for v in variaveis_escolhidas if v not in features]

if variaveis_invalidas:
    raise ValueError(f"Variáveis fora de features: {variaveis_invalidas}")

variacao_entrada = 0.10

resultados = []

for variavel in variaveis_escolhidas:
    linha_cenario = linha_base.copy()
    valor_original = linha_base[variavel]
    valor_alterado = valor_original * (1 + variacao_entrada)
    linha_cenario[variavel] = valor_alterado

    saida_nova = prever_linha(linha_cenario)
    variacao_saida = (saida_nova - saida_base) / saida_base
    indice_sensibilidade = variacao_saida / variacao_entrada

    resultados.append({
        "variável": variavel,
        "valor_original": valor_original,
        "valor_alterado": valor_alterado,
        "saída_original": saida_base,
        "saída_nova": saida_nova,
        "variação_saida_pct": variacao_saida * 100,
        "índice_sensibilidade": indice_sensibilidade,
        "impacto_absoluto_do_índice": abs(indice_sensibilidade),
    })

tabela_sensibilidade = pd.DataFrame(resultados)
tabela_sensibilidade


,variável,valor_original,valor_alterado,saída_original,saída_nova,variação_saida_pct,índice_sensibilidade,impacto_absoluto_do_índice
0,taxa_abandono_carrinho_pct,47.546,52.300,5.869,5.582,-4.891,-0.489,0.489
1,profundidade_scroll_pct,62.449,68.694,5.869,6.020,2.578,0.258,0.258


Compare os índices de sensibilidade e indique qual variável tem maior impacto sobre a taxa de conversão.

Mostre o raciocínio: cite os valores da tabela e explique o que eles significam para a decisão.

**Resposta:**

A variável com maior impacto entre as duas escolhidas é **taxa_abandono_carrinho_pct**. Quando a taxa de abandono aumenta 10%, a saída prevista cai de aproximadamente **5,87%** para **5,58%**, uma redução relativa de cerca de **4,89%** na conversão. O índice de sensibilidade fica em torno de **-0,489**.

Já a **profundidade_scroll_pct** também tem efeito relevante, mas menor. Quando ela aumenta 10%, a conversão prevista sobe de aproximadamente **5,87%** para **6,02%**, uma variação relativa de cerca de **2,58%**, com índice de sensibilidade próximo de **0,258**.

Portanto, a decisão de produto deve priorizar o abandono de carrinho, porque seu impacto absoluto sobre a conversão é quase duas vezes maior do que o impacto da profundidade de scroll. A profundidade de scroll continua sendo uma métrica útil para melhorar descoberta de produtos, mas o abandono parece ser a alavanca mais forte para ganho direto de conversão.


In [10]:
# Sensibilidade estendida: inclui todas as variáveis e também a direção de melhoria esperada.
# Para abandono e tempo de primeiro clique, melhorar significa reduzir 10%.
# Para profundidade de scroll, melhorar significa aumentar 10%.
direcao_melhoria = {
    "taxa_abandono_carrinho_pct": -0.10,
    "profundidade_scroll_pct": 0.10,
    "tempo_primeiro_clique_s": -0.10,
}

resultados_estendidos = []
for variavel in features:
    for nome_cenario, variacao in [("aumento_10pct", 0.10), ("melhoria_esperada", direcao_melhoria[variavel])]:
        linha_cenario = linha_base.copy()
        linha_cenario[variavel] = linha_base[variavel] * (1 + variacao)
        saida_nova = prever_linha(linha_cenario)
        resultados_estendidos.append({
            "variável": variavel,
            "cenário": nome_cenario,
            "variação_entrada_pct": variacao * 100,
            "saída_nova_pct": saida_nova,
            "delta_pontos_percentuais": saida_nova - saida_base,
            "variação_relativa_saída_pct": ((saida_nova - saida_base) / saida_base) * 100,
        })

tabela_sensibilidade_estendida = pd.DataFrame(resultados_estendidos)
display(tabela_sensibilidade_estendida)

fig = px.bar(
    tabela_sensibilidade_estendida.query("cenário == 'melhoria_esperada'"),
    x="variável",
    y="delta_pontos_percentuais",
    title="Ganho previsto de conversão em cenários de melhoria de 10%",
    text="delta_pontos_percentuais",
)
fig.update_traces(texttemplate="%{text:.3f} p.p.", textposition="outside")
fig.show()


,variável,cenário,variação_entrada_pct,saída_nova_pct,delta_pontos_percentuais,variação_relativa_saída_pct
0,taxa_abandono_carrinho_pct,aumento_10pct,10.0,5.582,-0.287,-4.891
1,taxa_abandono_carrinho_pct,melhoria_esperada,-10.0,6.156,0.287,4.891
2,profundidade_scroll_pct,aumento_10pct,10.0,6.020,0.151,2.578
3,profundidade_scroll_pct,melhoria_esperada,10.0,6.020,0.151,2.578
4,tempo_primeiro_clique_s,aumento_10pct,10.0,5.803,-0.066,-1.118
5,tempo_primeiro_clique_s,melhoria_esperada,-10.0,5.934,0.066,1.118


## Parte 4: Decisão

Recomende uma ação de produto ou interface com base na análise.

Sua recomendação deve citar os números da tabela de sensibilidade.

**Resposta:**

A recomendação é priorizar uma melhoria de interface voltada à **redução do abandono de carrinho**. Pela tabela de sensibilidade, um aumento de 10% no abandono reduz a conversão prevista em cerca de **4,89%** em termos relativos, com índice de sensibilidade de aproximadamente **-0,489**. Na direção contrária, uma redução de 10% no abandono elevaria a conversão prevista de aproximadamente **5,87%** para **6,16%**, ganho de cerca de **0,29 ponto percentual**.

A ação de produto recomendada seria simplificar o fluxo de carrinho e checkout: deixar custos e frete mais transparentes antes do fechamento, reduzir etapas desnecessárias, destacar benefícios de finalizar a compra e melhorar mensagens de erro ou indisponibilidade. A profundidade de scroll também deve ser acompanhada, pois um aumento de 10% nela eleva a conversão para cerca de **6,02%**, mas o impacto estimado é menor do que o efeito do abandono. Por isso, eu priorizaria abandono como frente principal e scroll como métrica secundária de acompanhamento.


Aponte uma limitação, risco ou hipótese da sua análise.

**Resposta:**

A maior limitação é que essa análise ajuda a priorizar, mas não prova causa e efeito. A taxa de abandono aparece como a variável mais importante no modelo, só que ela pode estar ligada a outros fatores que não estão na base, como preço, frete, campanhas, qualidade do tráfego, mix de produtos ou até algum problema técnico no checkout. Então eu usaria o resultado como uma hipótese forte para testar, não como uma certeza. Antes de escalar uma mudança grande, eu validaria em um experimento ou teste controlado, olhando não só conversão e abandono, mas também possíveis efeitos colaterais, como queda no ticket médio ou aumento de chamados de suporte.


In [11]:
# Cenários de decisão: impacto previsto de ações isoladas e combinadas.
cenarios = []

def adicionar_cenario(nome, ajustes):
    linha = linha_base.copy()
    for variavel, variacao in ajustes.items():
        linha[variavel] = linha_base[variavel] * (1 + variacao)
    previsao = prever_linha(linha)
    cenarios.append({
        "cenário": nome,
        "taxa_conversao_prevista_pct": previsao,
        "delta_pontos_percentuais": previsao - saida_base,
        "variação_relativa_pct": ((previsao - saida_base) / saida_base) * 100,
    })

adicionar_cenario("base média", {})
adicionar_cenario("reduzir abandono em 10%", {"taxa_abandono_carrinho_pct": -0.10})
adicionar_cenario("aumentar scroll em 10%", {"profundidade_scroll_pct": 0.10})
adicionar_cenario("reduzir tempo até clique em 10%", {"tempo_primeiro_clique_s": -0.10})
adicionar_cenario("melhoria combinada 10% nas 3 frentes", direcao_melhoria)
adicionar_cenario("melhoria realista 5% nas 3 frentes", {k: v / 2 for k, v in direcao_melhoria.items()})

tabela_cenarios = pd.DataFrame(cenarios)
display(tabela_cenarios)

fig = px.bar(
    tabela_cenarios,
    x="cenário",
    y="delta_pontos_percentuais",
    title="Impacto previsto por cenário de decisão",
    text="delta_pontos_percentuais",
)
fig.update_traces(texttemplate="%{text:.3f} p.p.", textposition="outside")
fig.show()


,cenário,taxa_conversao_prevista_pct,delta_pontos_percentuais,variação_relativa_pct
0,base média,5.869,0.000,0.000
1,reduzir abandono em 10%,6.156,0.287,4.891
2,aumentar scroll em 10%,6.020,0.151,2.578
3,reduzir tempo até clique em 10%,5.934,0.066,1.118
4,melhoria combinada 10% nas 3 frentes,6.373,0.504,8.587
5,melhoria realista 5% nas 3 frentes,6.121,0.252,4.294


## Ao Além dos Aléns

Faça uma simulação de Monte Carlo para estimar como a taxa de conversão pode variar sob incerteza nas variáveis de entrada.

In [12]:
# Simulação de Monte Carlo com incerteza nas variáveis de entrada e no erro residual do modelo.
# Além do cenário base, simulamos o cenário recomendado: redução média de 10% no abandono.
rng_mc = np.random.default_rng(2026)
n_simulacoes = 10000

desvios_incerteza = {
    "taxa_abandono_carrinho_pct": 5,
    "profundidade_scroll_pct": 8,
    "tempo_primeiro_clique_s": 1.5,
}

limites = {
    "taxa_abandono_carrinho_pct": (25, 75),
    "profundidade_scroll_pct": (25, 95),
    "tempo_primeiro_clique_s": (2, 15),
}

def simular_previsoes(medias, incluir_erro_modelo=True):
    amostras = pd.DataFrame({
        variavel: rng_mc.normal(medias[variavel], desvios_incerteza[variavel], n_simulacoes).clip(*limites[variavel])
        for variavel in features
    })
    amostras_design = np.column_stack([
        np.ones(len(amostras)),
        amostras[features].to_numpy(),
    ])
    previsoes_modelo = amostras_design @ coeficientes
    if incluir_erro_modelo:
        previsoes_modelo = previsoes_modelo + rng_mc.normal(0, rmse, n_simulacoes)
    return previsoes_modelo

medias_base = linha_base.copy()
medias_recomendacao = linha_base.copy()
medias_recomendacao["taxa_abandono_carrinho_pct"] *= 0.90

previsoes_base = simular_previsoes(medias_base)
previsoes_recomendacao = simular_previsoes(medias_recomendacao)

def resumo_simulacao(nome, previsoes):
    serie = pd.Series(previsoes)
    return {
        "cenário": nome,
        "média": serie.mean(),
        "desvio_padrão": serie.std(),
        "p10": serie.quantile(0.10),
        "p50_mediana": serie.quantile(0.50),
        "p90": serie.quantile(0.90),
        "prob_conversão_abaixo_5_5_pct": (serie < 5.5).mean() * 100,
        "prob_conversão_acima_6_0_pct": (serie > 6.0).mean() * 100,
    }

resumo_mc = pd.DataFrame([
    resumo_simulacao("base", previsoes_base),
    resumo_simulacao("recomendação: abandono -10%", previsoes_recomendacao),
])

resumo_mc


,cenário,média,desvio_padrão,p10,p50_mediana,p90,prob_conversão_abaixo_5_5_pct,prob_conversão_acima_6_0_pct
0,base,5.872,0.518,5.208,5.867,6.540,23.89,39.96
1,recomendação: abandono -10%,6.155,0.520,5.483,6.150,6.819,10.64,61.82


In [13]:
comparacao_mc = pd.DataFrame({
    "taxa_conversao_pct_prevista": np.concatenate([previsoes_base, previsoes_recomendacao]),
    "cenário": ["base"] * n_simulacoes + ["recomendação: abandono -10%"] * n_simulacoes,
})

fig = px.histogram(
    comparacao_mc,
    x="taxa_conversao_pct_prevista",
    color="cenário",
    nbins=45,
    barmode="overlay",
    opacity=0.65,
    title="Monte Carlo: distribuição simulada da taxa de conversão",
)
fig.show()


Interprete o que a distribuição simulada indica sobre o risco da sua recomendação.

**Resposta:**

A distribuição simulada mostra que reduzir abandono tende a melhorar a taxa de conversão prevista. No cenário base, a chance simulada de passar de **6,0%** de conversão ficou perto de **40%**. No cenário com redução de 10% no abandono, essa chance subiu para cerca de **62%**. Isso reforça a recomendação, porque mesmo colocando incerteza nas variáveis de entrada e erro do modelo, o cenário recomendado aparece melhor.

Mesmo assim, as distribuições ainda se sobrepõem. Ou seja, a melhoria não é garantia para todos os dias. Em alguns casos, por variação de tráfego, comportamento do usuário ou erro do próprio modelo, a conversão ainda pode ficar abaixo do esperado. Por isso, eu apresentaria essa recomendação como uma decisão bem sustentada pelos números, mas que ainda precisa ser testada em produção ou em um experimento controlado.


In [14]:
# Medida direta do ganho simulado: diferença entre as distribuições agregadas.
delta_medio_mc = previsoes_recomendacao.mean() - previsoes_base.mean()
delta_mediana_mc = np.median(previsoes_recomendacao) - np.median(previsoes_base)

pd.DataFrame({
    "métrica": ["ganho médio simulado", "ganho mediano simulado"],
    "delta_pontos_percentuais": [delta_medio_mc, delta_mediana_mc],
})


,métrica,delta_pontos_percentuais
0,ganho médio simulado,0.283
1,ganho mediano simulado,0.283


## Conclusão da análise

Com base nos resultados, eu priorizaria a redução da **taxa de abandono de carrinho**. Ela teve a maior correlação absoluta com a conversão e também o maior índice de sensibilidade: quando o abandono aumenta 10%, a conversão prevista cai cerca de **4,89%** em termos relativos. Na direção de melhoria, reduzir o abandono em 10% levaria a conversão prevista de aproximadamente **5,87%** para **6,16%**.

O Monte Carlo também reforça essa escolha. A probabilidade simulada de a conversão passar de **6,0%** sobe de cerca de **40%** no cenário base para aproximadamente **62%** no cenário com abandono 10% menor. Então minha recomendação seria atacar primeiro o fluxo de carrinho e checkout, mas tratando isso como hipótese de produto: o modelo indica onde testar primeiro, e não uma certeza causal já comprovada.


## Política de Uso de IA

O uso de IA é permitido para apoio técnico, revisão de texto e estudo dos conceitos.

As escolhas de variáveis, os cálculos, a comparação dos índices e a recomendação devem refletir sua análise dos resultados deste notebook.

Você deve ser capaz de explicar qualquer resposta entregue.

Respostas sem relação com os números gerados, com indícios de cópia ou que não possam ser justificadas poderão ser tratadas como fora da proposta.

## Instruções de entrega

A entrega deverá ser feita no GitHub ou no próprio Google Colab.

Links **sem permissão** de acesso terão um desconto de 20% na nota.